<a href="https://colab.research.google.com/github/Cyberpunk-San/ML-practice/blob/KNN/KNN_Scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

KNN Scratch

In [ ]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Load data
iris = load_iris()
X, y = iris.data, iris.target

# 2. Split and Scale
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
from collections import Counter

class KNNScratch:
    def __init__(self, k=3):
        self.k = k

    def fit(self, X, y):
        # KNN is a "Lazy Learner" - it just stores the data
        self.X_train = X
        self.y_train = y

    def predict(self, X):
        predictions = [self._predict(x) for x in X]
        return np.array(predictions)

    def _predict(self, x):
        # 1. Compute distances between x and all points in training set
        distances = [np.sqrt(np.sum((x - x_train)**2)) for x_train in self.X_train]

        # 2. Sort by distance and return indices of the first k neighbors
        k_indices = np.argsort(distances)[:self.k]

        # 3. Extract the labels of those k neighbors
        k_nearest_labels = [self.y_train[i] for i in k_indices]

        # 4. Return the most common label (Majority Vote)
        most_common = Counter(k_nearest_labels).most_common(1)
        return most_common[0][0]

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

# Using Petal Length and Petal Width
X_v = X_train_scaled[:, 2:4]
knn = KNNScratch(k=5)
knn.fit(X_v, y_train)

x_min, x_max = X_v[:, 0].min() - 1, X_v[:, 0].max() + 1
y_min, y_max = X_v[:, 1].min() - 1, X_v[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.1), np.arange(y_min, y_max, 0.1))

Z = knn.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

plt.figure(figsize=(10, 6))
plt.contourf(xx, yy, Z, alpha=0.3, cmap=ListedColormap(('red', 'green', 'blue')))
plt.scatter(X_v[:, 0], X_v[:, 1], c=y_train, edgecolors='k', cmap=ListedColormap(('red', 'green', 'blue')))
plt.title(f"KNN Decision Boundaries (k={knn.k})")
plt.show()

In [ ]:
import plotly.graph_objects as go

# Pick a random test point from X_test
test_pt = X_test_scaled[0, :3] # Use 3 features for 3D
train_pts = X_train_scaled[:, :3]

# Calculate distances manually for the plot
dists = np.sqrt(np.sum((train_pts - test_pt)**2, axis=1))
k_neighbors_idx = np.argsort(dists)[:5]

fig = go.Figure()

# Add training points
fig.add_trace(go.Scatter3d(x=train_pts[:,0], y=train_pts[:,1], z=train_pts[:,2],
                           mode='markers', marker=dict(size=3, color=y_train, opacity=0.5)))

# Add the test point
fig.add_trace(go.Scatter3d(x=[test_pt[0]], y=[test_pt[1]], z=[test_pt[2]],
                           mode='markers', marker=dict(size=8, color='black'), name="Test Point"))

# Draw lines to neighbors
for idx in k_neighbors_idx:
    fig.add_trace(go.Scatter3d(x=[test_pt[0], train_pts[idx,0]],
                               y=[test_pt[1], train_pts[idx,1]],
                               z=[test_pt[2], train_pts[idx,2]],
                               mode='lines', line=dict(color='black', width=2), showlegend=False))

fig.update_layout(title="KNN 3D: Test Point and its 5 Nearest Neighbors")
fig.show()

We implemented a Non-Parametric model. Unlike Regression or Naive Bayes, which try to summarize the data into a few numbers (weights or means), KNN keeps all the data.
1. The "Training": We simply stored the dataset.
2. The "Distance": We used Euclidean distance to find similarity.
3. The "Voting": By looking at the $K$ closest neighbors, we used a majority vote to decide the class.

Our 2D plot shows how KNN "partitions" the world into territories based on proximity, while the 3D plot visualizes the actual "search" process for neighbors.